# Rotterdam Landsat Surface temp calculation

In [14]:
# ==========================================
# STEP 1: INITIALIZATION & DEPENDENCIES
# ==========================================
# !pip install geemap ee  # Uncomment if you need to install them first

import ee
import geemap

# Initialize the Earth Engine library
try:
    ee.Initialize(project='applied-spatial-rotterdam')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='applied-spatial-rotterdam')

# ==========================================
# STEP 2: DEFINE COUPLING & MASKING FUNCTIONS
# ==========================================

# 1. Define the Area of Interest (AOI) bounding box around Rotterdam
rotterdam_aoi = ee.Geometry.Rectangle([3.9315, 51.8176, 4.6190, 52.0125])

# 2. Cloud masking function using the QA_PIXEL band
def mask_landsat_clouds(image):
    # Bits 3 and 4 correspond to Cloud and Cloud Shadow respectively
    qa = image.select('QA_PIXEL')
    cloud_shadow_mask = qa.bitwiseAnd(1 << 4).eq(0)
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0)
    
    # Combine masks
    mask = cloud_shadow_mask.And(cloud_mask)
    return image.updateMask(mask)

# 3. Scale Thermal Band 10 to Kelvin and convert directly to Celsius
# Formula: Celsius = (DN * 0.00341802) + 149.0 - 273.15
def apply_scale_factors(image):
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    
    
    lst_celsius = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)
    lst_celsius = lst_celsius.rename('LST_Celsius')
    
    return image.addBands(optical_bands, None, True).addBands(lst_celsius, None, True)

# ==========================================
# STEP 3: FILTER COLLECTION & PROCESS MEDIAN
# ==========================================

# Load Landsat 8 Collection 2 Tier 1 Level-2 data
landsat_collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(rotterdam_aoi)
    .filterDate('2025-05-01', '2025-08-31')
    .filter(ee.Filter.lt('CLOUD_COVER', 30))  # Exclude highly cloudy scenes
    .map(mask_landsat_clouds)
    .map(apply_scale_factors)
)

# Create a median composite of the LST band across the summer period
lst_composite = landsat_collection.select('LST_Celsius').median().clip(rotterdam_aoi)

# Quick print to see how many usable images were found in your timeframe
print(f"Number of images found in summer 2025: {landsat_collection.size().getInfo()}")

# ==========================================
# STEP 4: INTERACTIVE VISUALIZATION
# ==========================================

# Set up interactive map centered on Rotterdam
Map = geemap.Map(center=[51.9244, 4.4777], zoom=11)

# Define visualization parameters for LST (Celsius scale)
lst_vis = {
    'min': 15.0,   # Cooler surfaces (water bodies, green parks)
    'max': 35.0,   # Hotter surfaces (industrial zones, bare roofs)
    'palette': ['blue', 'green', 'yellow', 'orange', 'red']
}

# Add the LST layer and a colorbar map legend
Map.addLayer(lst_composite, lst_vis, 'Rotterdam Summer LST 2025')
Map.add_colorbar(lst_vis, label="Land Surface Temperature (°C)")

# Render the interactive map block
Map

# ==========================================
# STEP 5: EXPORT TO LOCAL PROCESSED DATA
# ==========================================

print("Exporting Rotterdam LST map locally...")
geemap.ee_export_image(
    lst_composite, 
    filename='../data/processed/rotterdam_lst_2025.tif', 
    scale=30,  # 30-meter resolution matching Landsat native thermal resolution
    region=rotterdam_aoi, 
    file_per_band=False
)

Number of images found in summer 2025: 6


Exporting Rotterdam LST map locally...
Generating URL ...
Please wait ...
Data downloaded to /Users/geethav/Downloads/Masters in Geomatics/Quarter 4/ARFW0501- Applied Spatial/Assignment/create-your-report-group-d/data/processed/rotterdam_lst_2025.tif


### NDVI calculation


In [15]:
# ==========================================
# STEP 1: INITIALIZATION & ACCOUNT BINDING
# ==========================================
import ee
import geemap
from IPython.display import display

# Change this to your active project ID string to bypass the 403 error permanently
MY_PROJECT_ID = 'applied-spatial-rotterdam' 

try:
    ee.Initialize(project=MY_PROJECT_ID)
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=MY_PROJECT_ID)

# ==========================================
# STEP 2: SPATIAL BOUNDS & IMAGE PROCESSING
# ==========================================

# 1. Bounding box around Rotterdam area of interest
rotterdam_aoi = ee.Geometry.Rectangle([3.9315, 51.8176, 4.6190, 52.0125])

# 2. Landsat 8 Cloud and Shadow Masking Function
def mask_landsat_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow_mask = qa.bitwiseAnd(1 << 4).eq(0)
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0)
    return image.updateMask(cloud_shadow_mask.And(cloud_mask))

# 3. Combined Scaling, LST, and NDVI Calculation Function
def process_landsat_indicators(image):
    # Scale optical bands (vital for accurate NDVI)
    optical = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    
    # Calculate NDVI using Normalized Difference: (Band 5 - Band 4) / (Band 5 + Band 4)
    # Band 5 = NIR, Band 4 = Red
    ndvi = optical.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    
    # Scale Thermal Band 10 and transform Kelvin directly to Celsius
    lst_celsius = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)
    lst_celsius = lst_celsius.rename('LST_Celsius')
    
    # Return image updated with clean, un-scaled parameters
    return image.addBands(optical, None, True).addBands(lst_celsius, None, True).addBands(ndvi)

# ==========================================
# STEP 3: DATA ENGINE AGGREGATION
# ==========================================

# Query, filter, and process the summer stack
landsat_collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(rotterdam_aoi)
    .filterDate('2025-06-01', '2025-08-31')
    .filter(ee.Filter.lt('CLOUD_COVER', 30))
    .map(mask_landsat_clouds)
    .map(process_landsat_indicators)
)

# Extract final median raster summaries for both layers
summer_composite = landsat_collection.median().clip(rotterdam_aoi)
lst_layer = summer_composite.select('LST_Celsius')
ndvi_layer = summer_composite.select('NDVI')

print(f"Successfully processed {landsat_collection.size().getInfo()} scenes for Rotterdam.")

# ==========================================
# STEP 4: INTERACTIVE TARGETED DISPLAY
# ==========================================

Map = geemap.Map(center=[51.9244, 4.4777], zoom=11)

# Color configurations
lst_vis = {
    'min': 15.0, 'max': 35.0,
    'palette': ['blue', 'green', 'yellow', 'orange', 'red']
}

ndvi_vis = {
    'min': -0.1, 'max': 0.7,
    'palette': ['#FFFFFF', '#CE7E45', '#DF923D', '#F1B555', '#FCD163', '#99B718', '#74A028', '#3E861A', '#206E1A', '#053C1A']
}

# Add both interactive spatial layers to map canvas switches
Map.addLayer(ndvi_layer, ndvi_vis, 'Rotterdam Summer NDVI 2025')
Map.addLayer(lst_layer, lst_vis, 'Rotterdam Summer LST 2025')

# Add legends so you can correlate green zones with cool spots
Map.add_colorbar(lst_vis, label="Land Surface Temperature (°C)")

# Render view window inside your notebook
display(Map)

# ==========================================
# STEP 5: FILE EXPORT TO CURRENT DIR STRUCTURE
# ==========================================
print("Saving output rasters to local directories...")

geemap.ee_export_image(
    lst_layer, 
    filename='../data/processed/rotterdam_lst_2025.tif', 
    scale=30, region=rotterdam_aoi, file_per_band=False
)

geemap.ee_export_image(
    ndvi_layer, 
    filename='../data/processed/rotterdam_ndvi_2025.tif', 
    scale=30, region=rotterdam_aoi, file_per_band=False
)
print("Done! Check your data/processed/ folder.")

Successfully processed 4 scenes for Rotterdam.


Map(center=[51.9244, 4.4777], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

Saving output rasters to local directories...
Generating URL ...
Please wait ...
Data downloaded to /Users/geethav/Downloads/Masters in Geomatics/Quarter 4/ARFW0501- Applied Spatial/Assignment/create-your-report-group-d/data/processed/rotterdam_lst_2025.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/geethav/Downloads/Masters in Geomatics/Quarter 4/ARFW0501- Applied Spatial/Assignment/create-your-report-group-d/data/processed/rotterdam_ndvi_2025.tif
Done! Check your data/processed/ folder.


### Local climate zones

In [16]:
# ==========================================
# STEP 6: EXTRACT LOCAL CLIMATE ZONES (LCZ)
# ==========================================
print("Extracting Local Climate Zones for Rotterdam...")

# 1. Load the RUBCLIM LCZ ImageCollection, reduce via mosaic, and clip to Rotterdam
# We use .mosaic() because it converts the collection down into a single composite image layer
lcz_dataset = (
    ee.ImageCollection('RUB/RUBCLIM/LCZ/global_lcz_map/latest')
    .mosaic()
    .clip(rotterdam_aoi)
)

# Select the recommended smoothed band
lcz_layer = lcz_dataset.select('LCZ_Filter')

# 2. Replicate official catalog visualization configurations
# Values 1-10 are Built environment classes; Values 11-17 are Natural classes
lcz_vis = {
    'bands': ['LCZ_Filter'],
    'min': 1,
    'max': 17,
    'palette': [
        '8c0000', # 1: Compact high-rise
        'd10000', # 2: Compact mid-rise
        'ff0000', # 3: Compact low-rise
        'bf4d00', # 4: Open high-rise
        'ff6600', # 5: Open mid-rise
        'ff9955', # 6: Open low-rise
        'faee05', # 7: Lightweight low-rise
        'bcbcbc', # 8: Large low-rise
        'ffccaa', # 9: Sparsely built
        '555555', # 10: Heavy industry
        '006a00', # 11: Dense trees (LCZ A)
        '00aa00', # 12: Scattered trees (LCZ B)
        '648525', # 13: Bush, scrub (LCZ C)
        'b9db79', # 14: Low plants (LCZ D)
        '000000', # 15: Bare rock or paved (LCZ E)
        'fbf7ae', # 16: Bare soil or sand (LCZ F)
        '6a6aff'  # 17: Water (LCZ G)
    ]
}

# Add the morphological LCZ classifications layer onto your existing Map widget canvas
Map.addLayer(lcz_layer, lcz_vis, 'Rotterdam Local Climate Zones (LCZ)')

# Refresh map view rendering in Jupyter
display(Map)

# ==========================================
# STEP 7: EXPORT LCZ RASTER TO DIRECTORY
# ==========================================
print("Saving LCZ raster locally...")

geemap.ee_export_image(
    lcz_layer, 
    filename='../data/processed/rotterdam_lcz_2018.tif', 
    scale=100,  # Note: LCZ native base resolution is 100 meters
    region=rotterdam_aoi, 
    file_per_band=False
)

print("Processing complete! You now have LST (30m), NDVI (30m), and LCZ (100m) synchronized arrays in data/processed/")

Extracting Local Climate Zones for Rotterdam...


Map(bottom=173659.0, center=[51.9244, 4.4777], controls=(WidgetControl(options=['position', 'transparent_bg'],…

Saving LCZ raster locally...
Generating URL ...
Please wait ...
Data downloaded to /Users/geethav/Downloads/Masters in Geomatics/Quarter 4/ARFW0501- Applied Spatial/Assignment/create-your-report-group-d/data/processed/rotterdam_lcz_2018.tif
Processing complete! You now have LST (30m), NDVI (30m), and LCZ (100m) synchronized arrays in data/processed/


# Adding details to Geopackage

In [10]:
import sys
!{sys.executable} -m pip install rasterstats

  Using cached rasterstats-0.20.0-py3-none-any.whl.metadata (4.2 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached click_plugins-1.1.1.2-py2.py3-none-any.whl.metadata (6.5 kB)
Using cached rasterstats-0.20.0-py3-none-any.whl (17 kB)
Using cached click_plugins-1.1.1.2-py2.py3-none-any.whl (11 kB)
  Created wheel for fiona: filename=fiona-1.10.1-cp314-cp314-macosx_10_15_universal2.whl size=1982235 sha256=444de7bcdf6b045d3e232be3b736f46ef89d360cb9317c5e654b3efe21a1d76b
  Stored in directory: /Users/geethav/Library/Caches/pip/wheels/3c/3e/56/f0e7dc93d0c0f5d2725b85aac3443c34ebc090c73bce308f16
Successfully built fiona
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [rasterstats]

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip3.14 install --upgrade pip


In [17]:
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
import os

# Define file paths based on your repository tree structure
gpkg_path = "../data/WijkBuurtkaart_2025_v1/wijkenbuurten_2025_v1.gpkg"
lst_raster = "../data/processed/rotterdam_lst_2025.tif"
ndvi_raster = "../data/processed/rotterdam_ndvi_2025.tif"
lcz_raster = "../data/processed/rotterdam_lcz_2018.tif"

output_gpkg = "../data/processed/rotterdam_wijkenbuurten_enriched.gpkg"

# ==========================================
# STEP 1: LOAD AND ALIGN VECTOR LAYERS
# ==========================================
print("Loading Geopackage layers...")
# Read the separate neighborhood (buurten) and district (wijken) layers
buurten_gdf = gpd.read_file(gpkg_path, layer="buurten")
wijken_gdf = gpd.read_file(gpkg_path, layer="wijken")

# Fetch the coordinate reference system (CRS) from one of the rasters to ensure alignment
with rasterio.open(lst_raster) as src:
    raster_crs = src.crs

# Match the vector projections to the raster's CRS (usually WGS84 EPSG:4326 for GEE exports)
if buurten_gdf.crs != raster_crs:
    print(f"Reprojecting vectors to match raster CRS: {raster_crs}")
    buurten_gdf = buurten_gdf.to_crs(raster_crs)
    wijken_gdf = wijken_gdf.to_crs(raster_crs)

# ==========================================
# STEP 2: CALCULATE ZONAL STATISTICS
# ==========================================
def extract_zonal_metrics(gdf, raster_path, metric, column_name):
    """Computes a specific raster statistic for each polygon in a GeoDataFrame."""
    print(f"Calculating {metric} from {os.path.basename(raster_path)}...")
    
    # zonal_stats returns a list of dictionaries containing the calculated metrics
    stats = zonal_stats(gdf, raster_path, stats=[metric])
    
    # Extract the values safely into a clean list
    extracted_values = [x[metric] if x else None for x in stats]
    
    # Map the list back onto the GeoDataFrame as a brand new column
    gdf[column_name] = extracted_values
    return gdf

# --- Process Buurten Layer ---
print("\n--- Processing Buurten (Neighborhoods) ---")
buurten_gdf = extract_zonal_metrics(buurten_gdf, lst_raster, 'mean', 'mean_LST_celsius')
buurten_gdf = extract_zonal_metrics(buurten_gdf, ndvi_raster, 'mean', 'mean_NDVI')
# For categorical data like LCZ, we want the 'majority' (most frequent class inside the zone)
buurten_gdf = extract_zonal_metrics(buurten_gdf, lcz_raster, 'majority', 'majority_LCZ')

# --- Process Wijken Layer ---
print("\n--- Processing Wijken (Districts) ---")
wijken_gdf = extract_zonal_metrics(wijken_gdf, lst_raster, 'mean', 'mean_LST_celsius')
wijken_gdf = extract_zonal_metrics(wijken_gdf, ndvi_raster, 'mean', 'mean_NDVI')
wijken_gdf = extract_zonal_metrics(wijken_gdf, lcz_raster, 'majority', 'majority_LCZ')

# ==========================================
# STEP 3: EXPORT ENRICHED LAYERS
# ==========================================
print(f"\nSaving enriched layers to {output_gpkg}...")

# Save both updated dataframes back into a fresh, unified GeoPackage file
buurten_gdf.to_file(output_gpkg, layer="buurten_enriched", driver="GPKG")
wijken_gdf.to_file(output_gpkg, layer="wijken_enriched", driver="GPKG")

print("Zonal statistics integration complete successfully!")

Loading Geopackage layers...
Reprojecting vectors to match raster CRS: EPSG:4326

--- Processing Buurten (Neighborhoods) ---
Calculating mean from rotterdam_lst_2025.tif...
Calculating mean from rotterdam_ndvi_2025.tif...
Calculating majority from rotterdam_lcz_2018.tif...

--- Processing Wijken (Districts) ---
Calculating mean from rotterdam_lst_2025.tif...
Calculating mean from rotterdam_ndvi_2025.tif...
Calculating majority from rotterdam_lcz_2018.tif...

Saving enriched layers to ../data/processed/rotterdam_wijkenbuurten_enriched.gpkg...
Zonal statistics integration complete successfully!
